# Fase 1 · Definición del problema y entorno reproducible

**Proyecto:** Fallas recurrentes de transmisión en la red de un operador móvil (NOC de WOM, octubre de 2026)
**Curso:** MCDI500 · Programación para la Ciencia de Datos · Magíster en Ciencia de Datos e IA, UNAB
**Integrante:** Edgar Arcia Velásquez (Grupo 6, trabajo individual) · **Docente:** Dr. Omar Salinas Silva

---

### Qué produce esta fase

La Fase 2 recibe un conjunto de datos y lo deja listo para analizar. Para eso, la Fase 1 deja resuelto:

1. **El problema escrito** como estructura de datos: pregunta, objetivos, alcance, restricciones y supuestos.
2. **Un entorno verificado** con evidencia: el intérprete del proyecto y las versiones de cada librería.
3. **La estructura del repositorio** y los archivos que lo hacen reproducible.
4. **Un módulo propio** (`src/utilidades.py`), probado con casos normales, límite y excepciones.
5. **La ficha y el diccionario del conjunto de datos**, con el rol analítico de cada variable.

> **Alcance de este notebook:** aquí no se limpia, no se imputa y no se transforma. Eso es la Fase 2.
> Esta fase solo **mide** lo que trae el archivo, para que la Fase 2 decida con cifras.

## Introducción

El centro de operaciones de red (NOC) de un operador móvil registra cada falla de la red como una
incidencia con número de ticket (TT). En octubre de 2026 se registraron **21.542 incidencias**.
Algunas se normalizan en minutos y otras tardan días.

Las fallas de **transmisión** son pocas, pero son las que más dejan sitios fuera de servicio. Un
sitio pierde su transmisión cuando se cae su enlace de microondas (MW), se corta la fibra propia
(FOP) o falla la fibra que arrienda a un proveedor. Si un mismo elemento de red falla una y otra
vez, el problema no se resuelve atendiendo cada incidencia por separado. Hay que escalarlo al área
que corresponde (transmisión propia o el proveedor) y evaluar un cambio de transmisión, por ejemplo
un enlace de respaldo o un cambio de medio. Para eso hace falta saber, con datos, **qué elementos
fallan más, cuánto tiempo acumulan sin servicio y qué tipo de enlace los deja caer**.

### Herramientas del ecosistema científico

| Pieza | Para qué se usa | Qué falla si se omite |
|---|---|---|
| Entorno virtual (`.venv`) | Aislar las versiones de las librerías del proyecto | El notebook da otro resultado en otro equipo |
| `requirements.txt` | Reconstruir el mismo entorno con un comando | Nadie puede reproducir las cifras |
| pandas y NumPy | Leer, medir y transformar la tabla | No hay forma práctica de procesar 21 mil filas |
| Jupyter | Juntar el texto, el código y la evidencia | El razonamiento queda separado del cálculo |
| Git y GitHub | Historial de cambios con autor, fecha y motivo | No se puede demostrar cómo avanzó el trabajo |

In [ ]:
import sys                      # intérprete en uso: se verifica en la sección 2
import json                     # exportación de metadatos
import platform                 # sistema operativo, para el registro del entorno
import subprocess               # consultas a Git desde el notebook (sección 6)
import importlib                # recarga del módulo propio
from datetime import date
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

import numpy as np
import pandas as pd

SEMILLA = 42                    # semilla fija: cualquier resultado aleatorio se puede repetir
np.random.seed(SEMILLA)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
print("Librerías cargadas. Semilla:", SEMILLA)

## 1. Definición del problema

**Qué hace este paso.** Deja el problema escrito como un **diccionario de Python** y no como un
párrafo suelto. Lo que el informe declara (problemática, objetivos, alcance y supuestos) sale de
aquí, y así el informe y el notebook no pueden contradecirse.

In [ ]:
PROYECTO = {
    "titulo": "Fallas recurrentes de transmisión en la red de un operador móvil",
    "grupo": "Grupo 6 (trabajo individual, informado al docente)",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": ["Edgar Arcia Velásquez"],
    "github": "edgararciav-cpu",
    "repositorio": "https://github.com/edgararciav-cpu/mcdi500-grupo6",
    "docente": "Dr. Omar Salinas Silva",
    "situacion": ("El NOC registró 21.542 incidencias en octubre de 2026; 588 son fallas de transmisión "
                  "(enlace de microondas, fibra propia o fibra de proveedor). Varios elementos de red fallaron "
                  "más de una vez en el mes, pero no hay una medición que diga cuáles concentran las fallas, "
                  "cuánto tiempo sin servicio acumulan y qué tipo de enlace las origina. Para el uptime cuentan "
                  "solo las incidencias con pérdida de servicio: criticidad P1, P2, P3 y P3E (P3 extendida, "
                  "las que pasan mucho tiempo fuera de servicio)."),
    "pregunta": ("¿Qué elementos de red concentran incidencias recurrentes de transmisión (microondas, fibra "
                 "propia FOP y fibra de proveedor), cuánto tiempo fuera de servicio acumulan y qué tipo de "
                 "enlace o proveedor las origina, para priorizar su escalamiento y evaluar cambios de "
                 "transmisión que mejoren el uptime?"),
    "objetivo_general": ("Construir un flujo reproducible que transforme el exporte de incidencias en un conjunto "
                         "limpio y validado, y en una tabla de recurrencia por elemento de red, apta para "
                         "priorizar las fallas de transmisión."),
    "objetivos_especificos": [
        "Convertir la duración (texto h:mm:ss) a minutos, tratar las duraciones inválidas y comparar su distribución y sus atípicos entre las fallas de transmisión y el resto.",
        "Clasificar cada falla de transmisión según el tipo de enlace (microondas, fibra propia FOP, fibra de proveedor o satelital) y marcar si afecta el uptime según su criticidad (P1, P2, P3 o P3E).",
        "Comparar al menos tres métodos para tratar las duraciones vacías, considerando que corresponden a incidencias aún no normalizadas.",
        "Corregir la codificación del texto, unificar las categorías escritas de varias formas y codificar la criticidad y el impacto con orden declarado.",
        "Entregar un conjunto validado con pruebas (assert) y una tabla de recurrencia por elemento de red (fallas con pérdida de servicio y minutos fuera de servicio acumulados), lista para las Fases 3 y 4.",
    ],
    "alcance": {
        "incluye": ["las 21.542 incidencias de octubre de 2026 se limpian completas",
                    "el análisis se enfoca en las 588 fallas de transmisión (Tx Propia y Tx Proveedor)",
                    "limpieza, transformación y validación (F1 y F2)"],
        "excluye": ["la decisión de cambiar la transmisión de un sitio (la toman las áreas de transmisión)",
                    "costos de enlaces o multas asociadas", "modelos predictivos (F3 y F4)",
                    "comparación con otros meses"],
    },
    "restricciones": [
        "Un solo mes de datos: la recurrencia se mide dentro del mes y no permite analizar estacionalidad.",
        "Datos internos anonimizados: los elementos de red y las personas aparecen como códigos.",
        "Las incidencias aún no normalizadas no tienen fecha de cierre ni duración.",
        "El tipo de enlace se deduce del texto de la incidencia NOC; no hay una columna que lo indique directo.",
    ],
    "supuestos": [
        "El exporte contiene todas las incidencias registradas en el mes.",
        "Una duración vacía corresponde a una incidencia aún no normalizada, no a un error de registro (se verifica en la sección 5.4).",
        "Las duraciones registradas son correctas: no hay forma de contrastarlas con otra fuente.",
        "El tipo de incidencia NOC (Tx Propia o Tx Proveedor) identifica bien las fallas de transmisión (su tamaño se mide en la sección 5.5).",
        "Solo las criticidades P1, P2, P3 y P3E corresponden a pérdida de servicio y afectan el uptime; P2 Redundancia, P2R y P4 no (definición operativa del NOC).",
    ],
    "criterios_exito": [
        "El notebook corre completo con Restart & Run All, sin errores.",
        "Cada decisión de la Fase 2 queda respaldada por una cifra del propio análisis.",
    ],
}
print("Proyecto definido:", PROYECTO["titulo"])

La función siguiente presenta el proyecto en pantalla. Recibe el diccionario **como parámetro** y
no lee la variable global. Así se puede reutilizar y probar por separado.

In [ ]:
def presentar_proyecto(config):
    '''Imprime la definición del proyecto de forma legible.

    Parámetros
    ----------
    config : dict
        Diccionario con la definición del proyecto (ver PROYECTO).
    '''
    print(config["titulo"].upper())
    print("=" * len(config["titulo"]))
    print(f"{config['asignatura']} · {config['grupo']}")
    print("Integrante:", ", ".join(config["integrantes"]), "· Docente:", config["docente"])
    print("\nSituación:\n ", config["situacion"])
    print("\nPregunta analizable:\n ", config["pregunta"])
    print("\nObjetivo general:\n ", config["objetivo_general"])
    print("\nObjetivos específicos:")
    for i, objetivo in enumerate(config["objetivos_especificos"], start=1):
        print(f"  {i}. {objetivo}")

presentar_proyecto(PROYECTO)

El alcance, las restricciones y los supuestos van aparte, porque cumplen otra función: delimitan lo
que **no** se hará y las condiciones bajo las que los resultados son válidos. Hay que distinguirlos:

- **Alcance:** lo que decide el proyecto, por ejemplo "no se comparan otros meses".
- **Restricción:** una condición impuesta por los datos, por ejemplo "solo hay un mes".

In [ ]:
for titulo, lista in [("Alcance · incluye", PROYECTO["alcance"]["incluye"]),
                      ("Alcance · excluye", PROYECTO["alcance"]["excluye"]),
                      ("Restricciones", PROYECTO["restricciones"]),
                      ("Supuestos", PROYECTO["supuestos"])]:
    print(titulo)
    for elemento in lista:
        print("  -", elemento)

## 2. Verificación del entorno

**Qué hace este paso.** Comprueba con evidencia, y no por confianza, que el notebook corre con el
**intérprete del proyecto** (`.venv`) y que cada librería instalada tiene la **misma versión** que
declara `requirements.txt`.

**Por qué es lo primero.** Si el kernel no es el del proyecto, todo lo que viene después puede dar
otro resultado en otro equipo. Es el origen de casi todos los "a mí me funciona" (lámina 34 de la
presentación de la Semana 1).

In [ ]:
def localizar_raiz(inicio=None):
    '''Devuelve la raíz del proyecto: la primera carpeta hacia arriba que contiene requirements.txt.

    Jupyter abre el notebook en su propia carpeta (F1/), no en la raíz; esta función permite usar
    rutas relativas a la raíz desde cualquier notebook.
    '''
    carpeta = Path(inicio or Path.cwd()).resolve()
    for candidata in [carpeta, *carpeta.parents]:
        if (candidata / "requirements.txt").is_file():
            return candidata
    raise FileNotFoundError("No encontré requirements.txt en esta carpeta ni en las superiores.")


def leer_requirements(ruta):
    '''Lee requirements.txt y devuelve {paquete: versión fijada}.'''
    versiones = {}
    for linea in Path(ruta).read_text(encoding="ascii").splitlines():
        if "==" in linea:
            paquete, ver = linea.strip().split("==", 1)
            versiones[paquete.lower()] = ver
    return versiones


RAIZ = localizar_raiz()
DECLARADAS = leer_requirements(RAIZ / "requirements.txt")

# Nombre en PyPI (requirements) -> nombre con que se importa
CLAVE = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "scikit-learn": "sklearn",
         "jupyterlab": None, "ipykernel": None}

def verificar_entorno(declaradas, claves):
    '''Compara la versión instalada de cada paquete clave con la declarada. Devuelve una tabla.'''
    filas = []
    for paquete, modulo in claves.items():
        try:
            instalada = version(paquete)
        except PackageNotFoundError:
            instalada = None
        if modulo:                      # además de estar instalado, debe poder importarse
            importlib.import_module(modulo)
        filas.append({"paquete": paquete, "declarada": declaradas.get(paquete),
                      "instalada": instalada, "coincide": instalada == declaradas.get(paquete)})
    return pd.DataFrame(filas)


print("Raíz del proyecto :", RAIZ)
print("Intérprete        :", sys.executable)
print("Python            :", platform.python_version(), "·", platform.system(), platform.release())
print("Paquetes fijados en requirements.txt:", len(DECLARADAS))
entorno = verificar_entorno(DECLARADAS, CLAVE)
entorno

In [ ]:
assert ".venv" in sys.executable, "El kernel no es el del proyecto: Kernel > Change Kernel > Python (mcdi500)"
assert entorno["coincide"].all(), "Hay librerías con una versión distinta a la declarada"
print("[OK] El notebook corre en .venv y las", len(entorno), "librerías clave coinciden con requirements.txt")

**Cómo leer este resultado.**

| Si se ve | Qué significa | Qué hacer |
|---|---|---|
| El intérprete no contiene `.venv` | El kernel no es el del proyecto | Kernel → Change Kernel → *Python (mcdi500)* |
| `coincide` en False | La versión instalada no es la declarada | `pip install -r requirements.txt` |
| Error al importar | La librería no está instalada en `.venv` | Instalar desde `requirements.txt` |

## 3. Estructura del repositorio y archivos de reproducibilidad

**Qué hace este paso.** Comprueba que existan las carpetas del proyecto y que `.gitignore` y
`requirements.txt` cumplan su función. No los crea: se crearon y versionaron antes, en los commits
`c7469f7` y `e005441`, y aquí se verifica que siguen correctos.

**Decisión documentada.** Los datos son comunes a todas las fases (`data/` en la raíz) y cada fase
tiene su carpeta con su notebook, igual que el proyecto modelo del curso.

In [ ]:
CARPETAS = ["F1", "F2", "F3", "F4", "data/raw", "data/processed", "docs", "src", "scripts"]
DIR_CRUDO, DIR_PROCESADO = RAIZ / "data" / "raw", RAIZ / "data" / "processed"
DIR_DOCS, DIR_SRC = RAIZ / "docs", RAIZ / "src"

estructura = pd.DataFrame([{"carpeta": c, "existe": (RAIZ / c).is_dir(),
                            "archivos": sum(1 for p in (RAIZ / c).glob("*") if p.is_file() and p.name != ".gitkeep")}
                           for c in CARPETAS])
gitignore = (RAIZ / ".gitignore").read_text(encoding="utf-8")
print(".gitignore excluye .venv/            :", ".venv/" in gitignore)
print(".gitignore excluye el exporte original:", "Export_Incidencias" in gitignore)
print("requirements.txt fija versiones con '==':", all("==" in l for l in (RAIZ / "requirements.txt").read_text(encoding="ascii").split()))
estructura

## 4. Módulos: funciones reutilizables en `src/`

**Qué hace este paso.** Importa el módulo propio `src/utilidades.py` y lo prueba.

**Por qué un módulo.** La Unidad 1 pide funciones, excepciones **y módulos**. El notebook documenta
y comunica; el módulo permite que la misma función se use en F2, F3 y F4 sin copiarla. Contiene:

| Elemento | Para qué |
|---|---|
| `reparar_texto` | Corrige tildes rotas por doble codificación (`EnergÃ­a` → `Energía`) |
| `normalizar_nombre`, `normalizar_columnas` | Nombres de columna en snake_case, sin tildes ni repetidos |
| `duracion_a_minutos` | Convierte `h:mm:ss` a minutos y rechaza formatos inválidos |
| `ConjuntoDatos` (clase) | Carga el archivo siempre con el mismo separador y codificación |
| `ProyectoError` (excepción propia) | Distingue un error de los datos de un error de pandas |

In [ ]:
if str(DIR_SRC) not in sys.path:
    sys.path.insert(0, str(DIR_SRC))
import utilidades
importlib.reload(utilidades)       # toma la última versión del archivo si se editó
print("Módulo importado desde:", Path(utilidades.__file__).relative_to(RAIZ))

### 4.1 Pruebas del módulo: casos normales, límite y excepciones

`assert` detiene el notebook si una condición no se cumple y deja constancia en la salida. Son los
mismos tres tipos de prueba que la rúbrica exige en la Fase 2 (criterio 11).

In [ ]:
# --- Casos normales ---
assert utilidades.reparar_texto("EnergÃ­a") == "Energía"
assert utilidades.normalizar_nombre("Tipo Incidencia NOC") == "tipo_incidencia_noc"
assert utilidades.normalizar_nombre("Tiempo de duraciÃ³n de la incidencia") == "tiempo_de_duracion_de_la_incidencia"
assert utilidades.duracion_a_minutos("1:30:00") == 90
assert utilidades.duracion_a_minutos("158:11:17") == 158 * 60 + 11 + 17 / 60
print("[OK] Casos normales")

# --- Casos límite ---
assert utilidades.reparar_texto("Cerrada") == "Cerrada"                  # texto sano: no se altera
assert utilidades.reparar_texto(None) is None                            # no es texto: se devuelve igual
assert utilidades.normalizar_nombre("</div>ï»¿TT") == "tt"               # la cabecera rota del exporte
assert utilidades.duracion_a_minutos("0:00:00") == 0
assert np.isnan(utilidades.duracion_a_minutos(""))                       # vacía = incidencia abierta
print("[OK] Casos límite")

# --- Excepciones: entradas inválidas deben fallar con el error propio ---
for entrada_invalida in ["1:30", "a:b:c", "1:75:00"]:
    try:
        utilidades.duracion_a_minutos(entrada_invalida)
        raise AssertionError(f"debió fallar con {entrada_invalida!r}")
    except utilidades.ProyectoError:
        pass
try:
    utilidades.normalizar_columnas(["Estado", "estado"])
    raise AssertionError("debió detectar nombres repetidos")
except utilidades.ProyectoError:
    pass
print("[OK] Excepciones: formatos inválidos y nombres repetidos se rechazan con ProyectoError")

## 5. Selección y ficha del conjunto de datos

**Qué hace este paso.** Documenta de dónde vienen los datos y los evalúa contra los requisitos del
curso, con las cifras medidas sobre el archivo real.

In [ ]:
FICHA = {
    "titulo": "Incidencias de red registradas por el NOC · octubre de 2026",
    "origen": "Exporte interno del sistema de tickets del NOC de WOM (operador móvil, Chile)",
    "permiso": "Uso autorizado por escrito por la jefatura; el docente está informado",
    "periodo": "2026-10-01 a 2026-10-31 (cierres hasta 2026-11-10)",
    "anonimizacion": "scripts/anonimizar_incidencias.py: personas, elementos de red, tickets y códigos TX reemplazados por códigos",
    "archivo": "data/raw/incidencias_oct2026_anon.csv",
    "separador": ";",
    "codificacion": "utf-8-sig",
}
DATOS = utilidades.ConjuntoDatos(RAIZ / FICHA["archivo"], sep=FICHA["separador"], encoding=FICHA["codificacion"])
df_crudo = DATOS.cargar()                     # lectura tal cual: los tipos los infiere pandas
df_texto = DATOS.cargar(como_texto=True)      # misma lectura, todo como texto, para auditar vacíos reales
print(DATOS)
print("Forma:", df_crudo.shape[0], "filas ×", df_crudo.shape[1], "columnas")

### 5.1 Evaluación contra los requisitos del curso

Requisitos **obligatorios** (descalifican si faltan) y **avisos** (lo que falte se declara como
restricción), según el anuncio del docente del 03-11 y `validar_dataset.py`.

In [ ]:
def evaluar_requisitos(df, ficha):
    '''Compara el conjunto con los requisitos del curso. Devuelve una tabla con el veredicto.'''
    pct_nulos = df.isna().mean() * 100
    fechas = [c for c in df.columns if "fecha" in utilidades.normalizar_nombre(c)]
    filas = [
        ("Al menos 2.000 filas", "obligatorio", len(df) >= 2000, f"{len(df):,} filas".replace(",", ".")),
        ("Al menos 12 columnas", "obligatorio", df.shape[1] >= 12, f"{df.shape[1]} columnas"),
        ("Al menos una variable numérica", "obligatorio", df.select_dtypes("number").shape[1] >= 1,
         f"{df.select_dtypes('number').shape[1]} numéricas al leer"),
        ("Al menos una variable categórica", "obligatorio", df.select_dtypes(exclude="number").shape[1] >= 1,
         f"{df.select_dtypes(exclude='number').shape[1]} de texto o categoría"),
        ("Presencia de valores faltantes", "aviso", pct_nulos.max() > 0, f"máximo {pct_nulos.max():.1f} %"),
        ("Variable de fecha", "aviso", len(fechas) > 0, f"{len(fechas)} columnas de fecha"),
        ("Archivo bajo 100 MB", "aviso", DATOS.ruta.stat().st_size < 100e6, f"{DATOS.ruta.stat().st_size / 1e6:.1f} MB"),
        ("Datos de los últimos 5 años", "obligatorio", True, ficha["periodo"]),
        ("Sin datos personales identificables", "obligatorio", True, "anonimizado (ver bitácora, paso 4)"),
    ]
    return pd.DataFrame(filas, columns=["criterio", "tipo", "cumple", "evidencia"])

evaluacion = evaluar_requisitos(df_crudo, FICHA)
assert evaluacion.loc[evaluacion["tipo"] == "obligatorio", "cumple"].all(), "Falla un requisito obligatorio"
evaluacion

### 5.2 Diccionario de variables

El diccionario declara el **rol analítico** de cada variable. El tipo que asigna pandas no basta:
`Q Network Element` es un número, pero `TT` es un identificador aunque también tenga dígitos. El
rol decide el tratamiento de la Fase 2.

El nombre original se conserva tal como viene en el exporte, con su codificación rota, y al lado va
el nombre normalizado que usará la Fase 2.

In [ ]:
ROLES = {
    "TT": ("identificador", "Número de ticket de la incidencia (no es único: un ticket puede abarcar varias filas)"),
    "Fecha de Inicio de incidencia": ("fecha", "Inicio de la incidencia (dd-mm-aaaa H:MM)"),
    "Fecha de Inicio ticket": ("fecha", "Apertura del ticket"),
    "Comuna": ("alta cardinalidad", "Comuna del elemento afectado"),
    "Estado": ("nominal", "Cerrada, Pendiente Automático, Abierta, Cancelada"),
    "Tipo Incidencia NOC": ("nominal", "Tipo de falla según el NOC (Energía, Tx, RF, Hardware, ...)"),
    "Incidencia NOC": ("nominal", "Subtipo de falla"),
    "Criticidad": ("ordinal", "P1 (más crítica) a P4. P1, P2, P3 y P3E (P3 extendida) son pérdida de servicio y afectan el uptime; P2 Redundancia, P2R y P4 no"),
    "Detalle": ("texto", "Descripción breve escrita por el operador"),
    "Impacto": ("ordinal", "Sin Impacto < Localizado < Regional < Amplio"),
    "Tiempo de Respuesta": ("discreta", "Tiempo comprometido de respuesta en horas (0, 2, 4, 24, 48)"),
    "Titulo": ("texto", "Título del ticket"),
    "Q Network Element": ("discreta", "Cantidad de elementos de red afectados"),
    "Network Element": ("identificador", "Elemento de red afectado (código anonimizado NE_)"),
    "Proveedor": ("nominal", "Proveedor asociado a la falla (eléctrico, TX, propietario)"),
    "Ticket de proveedor": ("identificador", "Ticket del proveedor (código TKP_ o marcador como N/A)"),
    "Incident Manager": ("nominal", "Responsable de la incidencia (código IM_)"),
    "Creador de TT": ("nominal", "Operador que creó el ticket (código OP_)"),
    "Fecha Cierre de incidencia": ("fecha", "Normalización de la incidencia"),
    "Fecha de Cierre de ticket": ("fecha", "Cierre del ticket"),
    "Tiempo de duración de la incidencia": ("continua", "Duración h:mm:ss; vacía si la incidencia no se ha normalizado"),
    "Tiempo de duración del ticket": ("continua", "Duración del ticket h:mm:ss"),
    "Código de Servicio": ("identificador", "Código de servicio (suministro eléctrico, no es RUT)"),
    "Tramo Afectado": ("texto", "Tramo de transmisión afectado; casi siempre NA o no aplica"),
    "Comunas": ("alta cardinalidad", "Comunas afectadas (puede listar varias)"),
    "Regiones": ("nominal", "Regiones afectadas en números romanos (puede listar varias)"),
    "Causa Raiz NOC": ("nominal", "Causa raíz según el NOC"),
    "Tipo Incidencia GMI": ("nominal", "Tipo según el área GMI"),
    "Incidenca GMI": ("nominal", "Subtipo según GMI (el nombre de la columna viene con error de tipeo)"),
    "Causa Raiz GMI": ("nominal", "Causa raíz según GMI"),
    "Horario Posible Degradación": ("vacía", "Sin datos en el exporte"),
    "ETR": ("fecha", "Hora estimada de restauración"),
    "Gestionable": ("binaria", "SI / NO"),
    "Gestor Uptime": ("binaria", "SI / NO"),
}
for i in range(1, 6):
    ROLES[f"Proveedor TX {i}"] = ("nominal", f"Proveedor del enlace de transmisión {i}")
    ROLES[f"Código servicio TX {i}"] = ("identificador", f"Código del servicio TX {i} (TXS_)")
    ROLES[f"Descripción TX {i}"] = ("texto", f"Descripción del enlace TX {i}")

def construir_diccionario(df, roles):
    '''Une el rol declarado de cada columna con lo que se mide en el archivo.'''
    filas = []
    for original in df.columns:
        legible = utilidades.reparar_texto(original)
        rol, descripcion = roles.get(legible, ("sin clasificar", ""))
        serie = df[original]
        filas.append({"variable": utilidades.normalizar_nombre(original), "nombre_original": original,
                      "rol_analitico": rol, "descripcion": descripcion, "dtype": str(serie.dtype),
                      "n_unicos": int(serie.nunique()), "pct_nulos": round(serie.isna().mean() * 100, 2)})
    return pd.DataFrame(filas)

diccionario = construir_diccionario(df_crudo, ROLES)
assert (diccionario["rol_analitico"] != "sin clasificar").all(), "Hay columnas sin rol declarado"
print("Variables por rol:", diccionario["rol_analitico"].value_counts().to_dict())
diccionario

### 5.3 Reconocimiento inicial (sin modificar nada)

**Qué hace este paso.** Mide los problemas que trae el archivo para que la Fase 2 decida con cifras.
Aquí **no se corrige nada**: solo se cuenta.

In [ ]:
def medir_problemas(df_txt):
    '''Cuenta los problemas de calidad visibles en el archivo leído como texto.'''
    celdas_rotas = df_txt.apply(lambda col: col.str.contains("Ã|Â", regex=True)).sum().sum()
    col_tramo = [c for c in df_txt.columns if utilidades.reparar_texto(c) == "Tramo Afectado"][0]
    variantes_na = df_txt[col_tramo][df_txt[col_tramo].str.strip().str.lower().isin(["na", "n/a", "no aplica"])].unique()
    col_tt = df_txt.columns[0]
    col_dur = [c for c in df_txt.columns if utilidades.reparar_texto(c) == "Tiempo de duración de la incidencia"][0]
    dur = df_txt[col_dur].str.strip()
    dur_invalida = dur.ne("") & ~dur.str.fullmatch(r"\d+:\d{2}:\d{2}")
    return pd.DataFrame([
        ("Duraciones con formato inválido (por ejemplo, negativas)", int(dur_invalida.sum())),
        ("Celdas con tildes rotas (Ã, Â)", int(celdas_rotas)),
        ("Formas de escribir 'no aplica' en Tramo Afectado", len(variantes_na)),
        ("Filas cuyo TT se repite en otra fila", int(df_txt[col_tt].duplicated(keep=False).sum())),
        ("Filas duplicadas exactas", int(df_txt.duplicated().sum())),
        ("Columnas con más de 60 % vacío", int((df_txt.eq("").mean() > 0.6).sum())),
    ], columns=["problema", "cantidad"])

problemas = medir_problemas(df_texto)
problemas

### 5.4 Verificación del supuesto sobre las duraciones vacías

El supuesto 2 de la sección 1 dice que una duración vacía es una incidencia **aún no normalizada**.
Se verifica cruzando el estado con la presencia de duración.

In [ ]:
col_estado = "Estado"
col_duracion = [c for c in df_texto.columns if utilidades.reparar_texto(c) == "Tiempo de duración de la incidencia"][0]
duracion_vacia = df_texto[col_duracion].str.strip().eq("")
cruce = pd.crosstab(df_texto[col_estado], duracion_vacia.map({True: "duración vacía", False: "con duración"}),
                    margins=True, margins_name="Total").rename_axis(columns="Duración")
print(f"Duraciones vacías: {duracion_vacia.sum():,} de {len(df_texto):,} ({duracion_vacia.mean():.1%})".replace(",", "."))
assert set(df_texto.loc[duracion_vacia, col_estado]) <= {"Abierta", "Pendiente Automatico"}, "Hay vacíos en incidencias cerradas"
print("[OK] Supuesto verificado: todas las duraciones vacías son incidencias Abiertas o Pendientes (no normalizadas)")
cruce

> **Consecuencia para la Fase 2.** El vacío **no ocurre al azar**: depende del estado. Rellenarlo con
> la media o la mediana inventaría una hora de cierre que no existe. La Fase 2 igual compara los
> métodos, porque la rúbrica lo pide, pero la decisión debe considerar este hecho.

### 5.5 Tamaño del foco de análisis: fallas de transmisión

**Qué hace este paso.** Cuenta cuántas incidencias son de transmisión, de qué subtipo y cuántos
elementos de red fallan más de una vez. Sirve para confirmar que la pregunta tiene datos suficientes
para responderse. Igual que en 5.3, aquí **solo se cuenta**: la clasificación por tipo de enlace y
la tabla de recurrencia con minutos se construyen en la Fase 2, con los datos ya limpios.

In [ ]:
TIPOS_TX = ["Tx Propia", "Tx Proveedor"]
es_tx = df_texto["Tipo Incidencia NOC"].isin(TIPOS_TX)
tx = df_texto[es_tx]
print(f"Fallas de transmisión: {len(tx)} de {len(df_texto):,} incidencias ({es_tx.mean():.1%})".replace(",", "."))
print("Por tipo:", tx["Tipo Incidencia NOC"].value_counts().to_dict())

CRITICIDAD_UPTIME = ["P1", "P2", "P3", "P3E"]          # pérdida de servicio: afectan el uptime
afecta_uptime = tx["Criticidad"].isin(CRITICIDAD_UPTIME)
print(f"Con pérdida de servicio (P1, P2, P3, P3E): {afecta_uptime.sum()} de {len(tx)} fallas de transmisión")

def medir_recurrencia(elementos):
    '''Cuenta cuántas veces falla cada elemento de red y resume la recurrencia.'''
    conteo = elementos.value_counts()
    return {"Elementos de red con al menos 1 falla": len(conteo),
            "Elementos con 2 o más fallas en el mes": int((conteo >= 2).sum()),
            "Elementos con 3 o más fallas en el mes": int((conteo >= 3).sum()),
            "Máximo de fallas de un mismo elemento": int(conteo.max())}

fallas_por_elemento = tx["Network Element"].value_counts()
fallas_uptime_por_elemento = tx.loc[afecta_uptime, "Network Element"].value_counts()
recurrencia = pd.DataFrame({"todas las fallas TX": medir_recurrencia(tx["Network Element"]),
                            "solo pérdida de servicio": medir_recurrencia(tx.loc[afecta_uptime, "Network Element"])})
assert (fallas_uptime_por_elemento >= 2).any(), "No hay recurrencia con pérdida de servicio: la pregunta no tendría datos"
print("[OK] Hay elementos con pérdidas de servicio repetidas: la pregunta tiene datos para responderse")
recurrencia

> **Por qué se separan las dos columnas.** Una falla de transmisión sin pérdida de servicio (por
> ejemplo, una pérdida de redundancia en P2 Redundancia o P4) no baja el uptime. Si se cuentan todas,
> aparecen arriba elementos que fallan seguido pero no dejan el sitio fuera de servicio. Por eso la
> tabla de recurrencia de la Fase 2 se construye con la columna de la derecha.

In [ ]:
# Mediana de duración por criticidad (solo filas con duración válida): la P3E es la que pasa más tiempo fuera
col_dur = [c for c in df_texto.columns if utilidades.reparar_texto(c) == "Tiempo de duración de la incidencia"][0]
dur_txt = df_texto[col_dur].str.strip()
valida = dur_txt.str.fullmatch(r"\d+:\d{2}:\d{2}")
minutos = dur_txt[valida].map(utilidades.duracion_a_minutos)
(minutos.groupby(df_texto.loc[valida, "Criticidad"].map(utilidades.reparar_texto))
        .agg(incidencias="size", mediana_min="median").round(0).sort_values("mediana_min", ascending=False))

In [ ]:
# Subtipos de falla: de este texto sale, en la Fase 2, el tipo de enlace (MW, FOP o fibra de proveedor)
pd.crosstab(tx["Incidencia NOC"].map(utilidades.reparar_texto),
            [tx["Tipo Incidencia NOC"], afecta_uptime.map({True: "con pérdida", False: "sin pérdida"})],
            margins=True, margins_name="Total").rename_axis(index="Incidencia NOC", columns=["Tipo", "Servicio"])

## 6. Control de versiones

**Qué hace este paso.** Consulta Git desde el notebook: la identidad configurada, la rama y el
historial. No ejecuta commits; los commits se hacen desde la terminal, con un mensaje que explica el
cambio.

In [ ]:
def git(*argumentos):
    '''Ejecuta un comando de Git en la raíz del proyecto. Devuelve (éxito, salida).'''
    r = subprocess.run(["git", *argumentos], cwd=RAIZ, capture_output=True, text=True, encoding="utf-8")
    return r.returncode == 0, (r.stdout or r.stderr).strip()

_, nombre = git("config", "user.name")
_, correo = git("config", "user.email")
_, rama = git("rev-parse", "--abbrev-ref", "HEAD")
_, historial = git("log", "--format=%h | %ad | %an | %s", "--date=format:%d-%m %H:%M")
print("Identidad de Git :", nombre, f"<{correo}>")
print("Rama actual      :", rama)
print("Historial:")
print(historial)

| Prefijo | Cuándo se usa | Ejemplo del historial |
|---|---|---|
| `docs:` | documentación | docs: crea estructura inicial del proyecto |
| `data:` | datos | data: incorpora incidencias de octubre 2026 anonimizadas en raw |
| `feat:` | algo nuevo que hace funcionar el proyecto | feat: define el entorno reproducible con requirements.txt |
| `fix:` | corrección | — |
| `test:` | pruebas | — |

## 7. Validación técnica de la fase

La función reúne las comprobaciones que la Fase 1 debe dejar cumplidas. Si alguna falla, el
notebook se detiene con un error que dice cuál.

In [ ]:
def validar_fase1(raiz, config, evaluacion, diccionario):
    '''Comprueba que la Fase 1 deja lo que exige la rúbrica. Lanza AssertionError si algo falla.'''
    comprobaciones = [
        ("Problema con pregunta, objetivo general y 3 a 5 objetivos específicos",
         bool(config["pregunta"]) and 3 <= len(config["objetivos_especificos"]) <= 5),
        ("Alcance, restricciones y supuestos declarados por separado",
         all(config[k] for k in ("alcance", "restricciones", "supuestos"))),
        ("Kernel del proyecto (.venv)", ".venv" in sys.executable),
        ("Carpetas del proyecto presentes", all((raiz / c).is_dir() for c in CARPETAS)),
        ("requirements.txt con versiones fijas", len(leer_requirements(raiz / "requirements.txt")) > 0),
        ("Módulo src/utilidades.py importable", (raiz / "src" / "utilidades.py").is_file()),
        ("Requisitos obligatorios del dataset cumplidos", evaluacion.query("tipo == 'obligatorio'")["cumple"].all()),
        ("Todas las variables con rol analítico", (diccionario["rol_analitico"] != "sin clasificar").all()),
    ]
    for texto, ok in comprobaciones:
        print(("[OK]   " if ok else "[FALLA]"), texto)
        assert ok, texto
    print(f"\nFase 1 validada: {len(comprobaciones)} de {len(comprobaciones)} comprobaciones")

validar_fase1(RAIZ, PROYECTO, evaluacion, diccionario)

## 8. Vinculación con el mapa conceptual

La rúbrica (criterio 14) pide relacionar cada elemento del mapa conceptual con lo implementado, en
los **seis tipos** de elemento. Hay que decir qué quedó implementado y qué queda proyectado para las
fases siguientes. Cada fila apunta a una evidencia que se puede verificar en el repositorio.

In [ ]:
VINCULACION = [
    # (elemento del mapa, tipo de elemento, estado, evidencia en el repositorio)
    ("Python + entorno virtual",              "herramientas",              "implementado", "Sección 2 · .venv y requirements.txt (commit e005441)"),
    ("Jupyter + kernel Python (mcdi500)",     "herramientas",              "implementado", "Sección 2 · kernel registrado y verificado"),
    ("Git y GitHub",                          "herramientas",              "implementado", "Sección 6 · historial con prefijos y una sola identidad"),
    ("Anonimización de los datos",            "procesos",                  "implementado", "scripts/anonimizar_incidencias.py · bitácora paso 4"),
    ("Exploración y medición de calidad",     "procesos",                  "implementado", "Secciones 5.3 y 5.4 de este notebook"),
    ("Foco en fallas de transmisión recurrentes", "decisiones técnicas",   "implementado", "Sección 1 (pregunta) y sección 5.5 (588 fallas, 264 con pérdida de servicio)"),
    ("Uptime = criticidad P1, P2, P3 y P3E",   "decisiones técnicas",     "implementado", "Sección 1 (supuesto 5) y sección 5.5"),
    ("Tabla de recurrencia por elemento de red", "procesos",               "proyectado",   "Fase 2 · clasificación MW, FOP y fibra de proveedor"),
    ("Limpieza, imputación y transformación", "procesos",                  "proyectado",   "Fase 2 · F2/F2_Preprocesamiento.ipynb"),
    ("Núcleo algorítmico y POO",              "procesos",                  "proyectado",   "Fase 3 · Sumativa 2"),
    ("raw → processed, rutas relativas, semilla 42", "flujo reproducible", "implementado", "Secciones 2 y 3 · localizar_raiz y SEMILLA"),
    ("Restart & Run All sin errores",         "flujo reproducible",        "implementado", "Numeración continua de este notebook"),
    ("Datos internos anonimizados con permiso", "decisiones técnicas",     "implementado", "Ficha 5 · bitácora pasos 3 y 4"),
    ("Vacíos de duración = incidencia abierta", "decisiones técnicas",     "implementado", "Sección 5.4 · supuesto verificado"),
    ("data/, docs/, src/, scripts/ y F1 a F4", "organización del repositorio", "implementado", "Sección 3 · commit c7469f7"),
    ("README generado desde el notebook",     "organización del repositorio", "implementado", "Sección 9 · README.md"),
    ("Celdas narrativas antes de cada bloque de código", "estructura del notebook", "implementado", "Todo este notebook"),
    ("Pruebas normal, límite y excepción",    "estructura del notebook",   "implementado", "Sección 4.1 · assert"),
]
vinculacion = pd.DataFrame(VINCULACION, columns=["elemento_mapa", "tipo", "estado", "evidencia"])
tipos_exigidos = {"herramientas", "procesos", "flujo reproducible", "decisiones técnicas",
                  "organización del repositorio", "estructura del notebook"}
assert tipos_exigidos <= set(vinculacion["tipo"]), "Falta algún tipo de elemento del mapa"
print("Elementos por estado:", vinculacion["estado"].value_counts().to_dict())
vinculacion

## 9. Persistencia y trazabilidad

**Qué hace este paso.** Guarda en `docs/` los productos de la fase, para que el informe los cite y
cualquiera los verifique. También genera el README desde el diccionario `PROYECTO`, así el README no
puede quedar desactualizado respecto del notebook (sugerencia del docente, lámina 68).

In [ ]:
diccionario.to_csv(DIR_DOCS / "diccionario_variables.csv", index=False, encoding="utf-8")
evaluacion.to_csv(DIR_DOCS / "evaluacion_criterios_dataset.csv", index=False, encoding="utf-8")
vinculacion.to_csv(DIR_DOCS / "vinculacion_mapa_conceptual.csv", index=False, encoding="utf-8")
problemas.to_csv(DIR_DOCS / "problemas_calidad_f1.csv", index=False, encoding="utf-8")

_, commit_actual = git("rev-parse", "--short", "HEAD")
metadatos = {
    "proyecto": PROYECTO["titulo"], "fase": "F1", "fecha_ejecucion": date.today().isoformat(),
    "commit_base": commit_actual, "semilla": SEMILLA, "python": platform.python_version(),
    "sistema": f"{platform.system()} {platform.release()}", "entorno_virtual": ".venv" in sys.executable,
    "dependencias": dict(zip(entorno["paquete"], entorno["instalada"])),
    "dataset": {"archivo": FICHA["archivo"], "filas": int(df_crudo.shape[0]), "columnas": int(df_crudo.shape[1]),
                "duraciones_vacias": int(duracion_vacia.sum()), "fallas_transmision": int(len(tx)),
                "elementos_tx_con_2_o_mas": int((fallas_por_elemento >= 2).sum()),
                "fallas_tx_con_perdida_de_servicio": int(afecta_uptime.sum()),
                "elementos_con_2_o_mas_perdidas": int((fallas_uptime_por_elemento >= 2).sum())},
}
(DIR_DOCS / "metadatos_fase1.json").write_text(json.dumps(metadatos, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(metadatos, ensure_ascii=False, indent=2))

In [ ]:
def generar_readme(config, ficha, metadatos, ruta):
    '''Escribe el README técnico a partir de la definición del proyecto.'''
    objetivos = "\n".join(f"{i}. {o}" for i, o in enumerate(config["objetivos_especificos"], 1))
    filas_txt = f"{metadatos['dataset']['filas']:,}".replace(",", ".")   # 21.542: separador de miles chileno
    texto = f'''# {config["titulo"]}

Proyecto del curso **{config["asignatura"]}** · Magíster en Ciencia de Datos e Inteligencia Artificial, Universidad Andrés Bello.

- **Integrante:** {", ".join(config["integrantes"])} ([@{config["github"]}](https://github.com/{config["github"]})) · {config["grupo"]}
- **Docente:** {config["docente"]}

## Problemática

{config["situacion"]}

**Pregunta:** {config["pregunta"]}

**Objetivo general:** {config["objetivo_general"]}

**Objetivos específicos:**

{objetivos}

## Datos

- **Origen:** {ficha["origen"]}. {ficha["permiso"]}.
- **Período:** {ficha["periodo"]}.
- **Archivo:** `{ficha["archivo"]}` ({filas_txt} filas × {metadatos["dataset"]["columnas"]} columnas, separador `{ficha["separador"]}`).
- **Anonimización:** {ficha["anonimizacion"]}. El exporte original no forma parte del repositorio.

## Estructura

```
data/raw/        datos de entrada anonimizados (no se modifican)
data/processed/  resultado de la limpieza y transformación (F2)
docs/            diccionario de variables, bitácora, metadatos y vinculación con el mapa
src/             módulo utilidades.py (funciones y clases reutilizables)
scripts/         preparación previa de los datos (anonimización)
F1/ F2/ F3/ F4/  un notebook por fase
```

## Cómo ejecutar el proyecto desde cero (Windows, PowerShell)

```
git clone {config["repositorio"]}.git
cd mcdi500-grupo6
python -m venv .venv
.\\.venv\\Scripts\\python.exe -m pip install -r requirements.txt
.\\.venv\\Scripts\\python.exe -m ipykernel install --user --name mcdi500 --display-name "Python (mcdi500)"
.\\.venv\\Scripts\\python.exe -m jupyter lab
```

En JupyterLab, abrir los notebooks en orden (`F1/F1_Definicion.ipynb`, luego `F2/F2_Preprocesamiento.ipynb`), elegir el kernel **Python (mcdi500)** y ejecutar *Kernel → Restart Kernel and Run All Cells*. En macOS o Linux se usa `.venv/bin/python` en lugar de `.\\.venv\\Scripts\\python.exe`.

## Reproducibilidad

- Python {metadatos["python"]} en entorno virtual `.venv`; versiones fijas en `requirements.txt`.
- Semilla aleatoria: {metadatos["semilla"]}. Rutas relativas a la raíz del proyecto (`localizar_raiz`).
- Cada notebook verifica el entorno en su primera sección y se detiene si algo no coincide.

## Convención de commits

`docs:` documentación · `data:` datos · `feat:` código nuevo · `fix:` corrección · `test:` pruebas. El historial con el detalle de cada commit y el criterio de la rúbrica que respalda está en `docs/bitacora_proceso.md`.

## Uso de herramientas de inteligencia artificial

Se utilizó un asistente de inteligencia artificial generativa como apoyo para explicar conceptos, revisar código y redactar borradores. Las decisiones, la ejecución y la revisión final son del autor.
'''
    ruta.write_text(texto, encoding="utf-8")
    return ruta

print("README generado:", generar_readme(PROYECTO, FICHA, metadatos, RAIZ / "README.md").relative_to(RAIZ))

## 10. Conclusiones y reflexión técnica

- **El conjunto cumple los requisitos del curso** (sección 5.1) y trae problemas reales que dan
  material a la Fase 2: miles de celdas con tildes rotas, la misma categoría escrita de varias
  formas, columnas casi vacías y tickets repetidos (sección 5.3).
- **El hallazgo más importante es el de las duraciones vacías** (sección 5.4). No son datos perdidos
  al azar: son incidencias que aún no se normalizan. Eso condiciona la Fase 2, porque rellenarlas
  con un valor central inventaría cierres que no ocurrieron.
- **La pregunta tiene datos para responderse** (sección 5.5): hay 588 fallas de transmisión; 264
  dejaron el servicio caído (P1, P2, P3 o P3E) y 35 elementos de red perdieron servicio 2 o más
  veces en el mes. Esa recurrencia es la que justifica escalar con las áreas de transmisión y
  evaluar cambios de enlace.
- **Contar solo la pérdida de servicio cambia la lista.** El elemento con más fallas de transmisión
  tiene 19, pero solo 1 dejó el servicio caído; las otras fueron pérdidas de redundancia o P4. Sin
  ese filtro se escalarían sitios equivocados.
- **Hay 2 duraciones negativas** en incidencias cerradas (sección 5.3). La Fase 2 decide cómo
  tratarlas antes de calcular minutos acumulados.
- **El entorno quedó verificado con evidencia**: el kernel del proyecto y las versiones que coinciden
  con `requirements.txt`. El módulo propio quedó probado con casos normales, límite y excepciones.

**Queda proyectado:** la limpieza, la imputación, la transformación y el escalamiento (Fase 2); el
núcleo algorítmico (Fase 3); la visualización y la comunicación (Fase 4).

## 11. Verificación antes de entregar

- [ ] Corre completo tras *Kernel → Restart Kernel and Run All Cells*, sin errores.
- [ ] La numeración de ejecución es continua (1, 2, 3, ...).
- [ ] Cada bloque de código está precedido por una celda de texto que explica qué hace y por qué.
- [ ] No limpia, no imputa y no transforma: solo define, verifica y mide.
- [ ] Las salidas quedaron guardadas: no se borran antes de entregar.
- [ ] Los archivos de `docs/` y el README se versionan en el mismo commit que este notebook.